# Разбор надежной небольшой группы объектов класса `O`

Этот ноутбук смотрит только на небольшую группу объектов, которые после исправления парсера все еще остаются в надежной группе `O`. Здесь мы не разбираем спорный пул `OB`: он уже вынесен в отдельный контур проверки.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    # Ищем корень репозитория по каталогам .git и src.
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Не удалось определить корень репозитория из текущей рабочей директории.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


## Что делаем

- проверяем, какие исходные спектральные обозначения остались в надежной группе `O`;
- смотрим, что говорит по ним горячий модуль Gaia `ESP-HS`;
- оцениваем, похожа ли эта группа на небольшой, но правдоподобный хвост реальных `O`-звезд;
- сравниваем ее с уже имеющимися чистыми эталонными наборами `O` и `B`.


In [2]:
# Загружаем рабочие таблицы обзора и эталонные сводки сравнения.
from exohost.reporting.coarse_secure_o_tail_review import (
    build_secure_o_tail_esphs_frame,
    build_secure_o_tail_raw_label_frame,
    build_secure_o_tail_star_frame,
    build_secure_o_tail_summary_frame,
    load_secure_o_tail_review_frame,
)
from exohost.reporting.coarse_secure_o_reference_comparison import (
    build_secure_o_reference_distance_frame,
    build_secure_o_reference_group_summary_frame,
    load_secure_o_reference_comparison_frame,
)
from exohost.reporting.notebook_display import rename_frame_for_display, scalar_to_int
from exohost.reporting.notebook_labels import BOOLEAN_LABELS, REFERENCE_GROUP_LABELS

review_df = load_secure_o_tail_review_frame(dotenv_path='.env')
summary_df = build_secure_o_tail_summary_frame(review_df)
raw_label_df = build_secure_o_tail_raw_label_frame(review_df)
esphs_df = build_secure_o_tail_esphs_frame(review_df)
star_df = build_secure_o_tail_star_frame(review_df)
comparison_df = load_secure_o_reference_comparison_frame(dotenv_path='.env')
comparison_summary_df = build_secure_o_reference_group_summary_frame(comparison_df)
comparison_distance_df = build_secure_o_reference_distance_frame(comparison_df)


In [3]:
# Показываем сводку по размеру группы и покрытию ключевых полей.
display(
    rename_frame_for_display(
        summary_df,
        column_mapping={
            'n_rows': 'Число строк',
            'n_with_numeric_subclass': 'Строк с числовым подклассом',
            'n_with_teff_esphs': 'Строк с температурой `ESP-HS`',
            'n_in_gold_sample_oba': 'Строк в выборке OBA Gaia',
            'median_teff_gspphot': 'Медианная температура `GSP-Phot`',
            'median_teff_esphs': 'Медианная температура `ESP-HS`',
            'median_lum_flame': 'Медианная светимость FLAME',
        },
    )
)


,Число строк,Строк с числовым подклассом,Строк с температурой `ESP-HS`,Строк в выборке OBA Gaia,Медианная температура `GSP-Phot`,Медианная температура `ESP-HS`,Медианная светимость FLAME
0,11,2,1,2,15522.557617,33647.496,1096.2709


In [4]:
# Смотрим, какие исходные спектральные обозначения остались в надежной группе `O`.
display(
    rename_frame_for_display(
        raw_label_df,
        column_mapping={
            'raw_sptype': 'Исходное спектральное обозначение',
            'n_rows': 'Число строк',
            'share': 'Доля',
        },
    )
)


,Исходное спектральное обозначение,Число строк,Доля
0,O,2,0.181818
1,O4V,1,0.090909
2,O(He),1,0.090909
3,O(H)3/4,1,0.090909
4,O(H):III/V(e),1,0.090909
5,O(H)9I,1,0.090909
6,O8I/III,1,0.090909
7,O7.5I/III,1,0.090909
8,O(H)4/5V,1,0.090909
9,O(H)8If,1,0.090909


In [5]:
# Сверяемся с горячим модулем Gaia `ESP-HS`.
display(
    rename_frame_for_display(
        esphs_df,
        column_mapping={
            'spectraltype_esphs': 'Класс `ESP-HS`',
            'flags_esphs': 'Флаги `ESP-HS`',
            'has_teff_esphs': 'Есть температура `ESP-HS`',
            'n_rows': 'Число строк',
        },
        value_mapping={'has_teff_esphs': BOOLEAN_LABELS},
    )
)


,Класс `ESP-HS`,Флаги `ESP-HS`,Есть температура `ESP-HS`,Число строк
0,O,92.0,Нет,4
1,O,91.0,Нет,3
2,O,93.0,Нет,2
3,O,1.0,Да,1
4,O,95.0,Нет,1


In [6]:
# Показываем сами объекты с физикой и признаками из Gaia.
display(
    rename_frame_for_display(
        star_df,
        column_mapping={
            'source_id': 'source_id',
            'raw_sptype': 'Исходное спектральное обозначение',
            'spectral_subclass': 'Спектральный подкласс',
            'spectraltype_esphs': 'Класс `ESP-HS`',
            'esphs_class_letter': 'Буква класса `ESP-HS`',
            'flags_esphs': 'Флаги `ESP-HS`',
            'in_gold_sample_oba_stars': 'Входит в выборку OBA Gaia',
            'teff_gspphot': 'Температура `GSP-Phot`',
            'teff_esphs': 'Температура `ESP-HS`',
            'logg_gspphot': '`logg` `GSP-Phot`',
            'logg_esphs': '`logg` `ESP-HS`',
            'radius_flame': 'Радиус FLAME',
            'lum_flame': 'Светимость FLAME',
        },
        value_mapping={'in_gold_sample_oba_stars': BOOLEAN_LABELS},
    )
)


,source_id,Исходное спектральное обозначение,Спектральный подкласс,Класс `ESP-HS`,Буква класса `ESP-HS`,Флаги `ESP-HS`,Входит в выборку OBA Gaia,Температура `GSP-Phot`,Температура `ESP-HS`,`logg` `GSP-Phot`,`logg` `ESP-HS`,Радиус FLAME,Светимость FLAME
0,443815467670295680,O,NaN,O,O,91.0,Нет,10225.365234,NaN,2.9870,NaN,11.566925,1316.769700
1,1964942177804409728,O,NaN,O,O,93.0,Нет,12425.124023,NaN,4.4983,NaN,1.322884,37.944633
2,2168667564882421376,O4V,4.0,O,O,1.0,Да,15031.999023,33647.496,3.4447,3.547713,7.860304,2860.144800
3,2299420113258231808,O(He),NaN,O,O,91.0,Нет,18382.367188,NaN,4.5747,NaN,0.283203,8.511230
4,4085994773196437888,O(H)3/4,NaN,O,O,92.0,Нет,19995.814453,NaN,4.3500,NaN,2.794759,1124.987700
5,4292267621344388864,O(H):III/V(e),NaN,O,O,92.0,Нет,15557.005859,NaN,3.3584,NaN,4.630312,1096.270900
6,4514868732516293760,O(H)9I,NaN,O,O,92.0,Да,15049.850586,NaN,3.8660,NaN,4.237189,832.551330
7,5337243678528184704,O8I/III,8.0,O,O,92.0,Нет,11230.172852,NaN,2.8761,NaN,12.935532,2397.910000
8,5337610159501805952,O7.5I/III,NaN,O,O,93.0,Нет,15522.557617,NaN,3.5327,NaN,6.913195,2516.133500
9,5348444781415074432,O(H)4/5V,NaN,O,O,91.0,Нет,19834.074219,NaN,4.5516,NaN,1.910222,513.143500


## Сравнение с чистыми эталонными наборами `O` и `B`

Ниже сравниваем эти `11` объектов не с произвольными данными, а с уже существующими чистыми наборами `O` и `B` из `public`.

Важно: это не новый внешний источник, а удобный эталон, чтобы понять, к какому полюсу ближе текущая небольшая надежная группа `O`.


In [7]:
# Сравниваем группу с чистыми эталонными наборами `O` и `B`.
display(
    rename_frame_for_display(
        comparison_summary_df,
        column_mapping={
            'comparison_group': 'Сравниваемая группа',
            'n_rows': 'Число строк',
            'median_teff_gspphot': 'Медианная температура `GSP-Phot`',
            'median_logg_gspphot': 'Медианный `logg` `GSP-Phot`',
            'median_radius_gspphot': 'Медианный радиус `GSP-Phot`',
            'median_parallax': 'Медианный параллакс',
        },
        value_mapping={'comparison_group': REFERENCE_GROUP_LABELS},
    )
)


,Сравниваемая группа,Число строк,Медианная температура `GSP-Phot`,Медианный `logg` `GSP-Phot`,Медианный радиус `GSP-Phot`,Медианный параллакс
0,Чистый эталон `B`,3000,12825.095215,4.20320,2.14295,0.176296
1,Эталон эволюционировавших `B`,3000,12355.913574,3.77930,4.0046,0.256354
2,Эталон эволюционировавших `O`,3000,34604.617188,3.89625,8.79655,0.138732
3,Чистый эталон `O`,3000,30145.717773,4.02485,6.38505,0.090535
4,Текущий надежный хвост `O`,11,15522.557617,3.86600,<NA>,0.245577


In [8]:
# Смотрим, к какому эталону эта группа ближе по основным физическим величинам.
display(
    rename_frame_for_display(
        comparison_distance_df,
        column_mapping={
            'comparison_group': 'Сравниваемая группа',
            'abs_diff_teff_gspphot': 'Разница по температуре `GSP-Phot`',
            'abs_diff_logg_gspphot': 'Разница по `logg` `GSP-Phot`',
            'abs_diff_radius_gspphot': 'Разница по радиусу `GSP-Phot`',
            'abs_diff_parallax': 'Разница по параллаксу',
        },
        value_mapping={'comparison_group': REFERENCE_GROUP_LABELS},
    )
)


,Сравниваемая группа,Разница по температуре `GSP-Phot`,Разница по `logg` `GSP-Phot`,Разница по радиусу `GSP-Phot`,Разница по параллаксу
0,Чистый эталон `B`,2697.462402,0.33720,<NA>,0.069281
1,Эталон эволюционировавших `B`,3166.644043,0.08670,<NA>,0.010777
2,Чистый эталон `O`,14623.160156,0.15885,<NA>,0.155042
3,Эталон эволюционировавших `O`,19082.059570,0.03025,<NA>,0.106845


In [9]:
findings_df = pd.DataFrame(
    [
        {'Показатель': 'Число строк', 'Значение': scalar_to_int(summary_df.loc[0, 'n_rows'])},
        {
            'Показатель': 'Строк с числовым подклассом',
            'Значение': scalar_to_int(summary_df.loc[0, 'n_with_numeric_subclass']),
        },
        {
            'Показатель': 'Строк с `teff_esphs`',
            'Значение': scalar_to_int(summary_df.loc[0, 'n_with_teff_esphs']),
        },
        {
            'Показатель': 'Строк в выборке OBA Gaia',
            'Значение': scalar_to_int(summary_df.loc[0, 'n_in_gold_sample_oba']),
        },
        {
            'Показатель': 'Самое частое исходное обозначение',
            'Значение': str(raw_label_df.loc[0, 'raw_sptype']) if not raw_label_df.empty else 'нет данных',
        },
        {
            'Показатель': 'Ближайшая эталонная группа',
            'Значение': str(comparison_distance_df.loc[0, 'comparison_group']) if not comparison_distance_df.empty else 'нет данных',
        },
    ]
)
display(findings_df)


,Показатель,Значение
0,Число строк,11
1,Строк с числовым подклассом,2
2,Строк с `teff_esphs`,1
3,Строк в выборке OBA Gaia,2
4,Самое частое исходное обозначение,O
5,Ближайшая эталонная группа,reference_b


## Краткий вывод

- эта группа уже не похожа на старый спорный пул `OB...`;
- в исходных обозначениях здесь действительно остаются формы `O`, `O4V`, `O8I/III` и похожие записи;
- горячий модуль Gaia сохраняет для этих объектов букву класса `O`, но численная температура `teff_esphs` заполнена только у одного объекта;
- по `teff_gspphot` эта небольшая группа заметно ближе к эталонному `B`, чем к эталонному `O`.

Итог: эти `11` объектов уже нельзя считать мусором после разбора меток, но и автоматически расширять по ним класс `O` пока нельзя.
